In [ ]:
import datetime as _dt
import os

import pandas as pd
from IPython.display import HTML, Image
from IPython.display import display as ipy_display
from ugbio_ppmseq.ppmSeq_utils import _assert_adapter_version_supported

In [ ]:
# papermill parameters
sample_name = None
adapter_version = None  # accepted but intentionally not displayed in the report
statistics_h5 = None
strand_ratio_category_png = None
strand_ratio_category_concordance_png = None
sr_hist_png = None
sr_by_et_png = None
read_length_png = None
read_length_by_st_png = None
logo_file = None
trimmer_failure_codes_csv = None
sorter_stats_csv = None
extra_info = {}

In [ ]:
if statistics_h5 is None or adapter_version is None or strand_ratio_category_png is None:
    raise ValueError("Missing required input files")
_assert_adapter_version_supported(adapter_version)
sample_name = sample_name or "(unnamed sample)"
# extra_info may arrive as a plain dict from the CLI or as a stringified dict from papermill;
# coerce to a dict either way.
if extra_info is None:
    extra_info = {}
if isinstance(extra_info, str):
    import ast

    extra_info = ast.literal_eval(extra_info) if extra_info else {}

In [ ]:
IMAGE_WIDTH = 900


def _show_logo(path, align="center"):
    if path and os.path.isfile(path):
        with open(path) as f:
            encoded = f.read().strip()
        ipy_display(
            HTML(
                f'<div style="text-align:{align};margin:12px 0;">'
                f'<img src="data:image/png;base64,{encoded}" style="max-width:320px;height:auto;"/>'
                f"</div>"
            )
        )


def _sample_header():
    # Printed above every table so the reader can always tell which sample the numbers
    # in the row below refer to.
    ipy_display(HTML(f'<p style="font-weight:bold;margin:2px 0;">Sample: {sample_name}</p>'))


def _caption(label, text):
    # Italic caption line under a figure or table, matching the srsnv report layout.
    ipy_display(HTML(f'<p style="font-style:italic;margin:4px 0 16px 0;"><b>{label}.</b> {text}</p>'))


def _show_image(path, caption_label, caption_text, width=IMAGE_WIDTH):
    if path and os.path.isfile(path):
        ipy_display(Image(path, width=width))
        _caption(caption_label, caption_text)

In [ ]:
_show_logo(logo_file, align="left")
today = _dt.date.today().isoformat()
header_html = (
    f'<div style="text-align:left;">'
    f'<div style="font-size:14px;color:#888;">{today}</div>'
    f'<div style="font-size:36px;font-weight:bold;margin-top:2px;">ppmSeq QC report</div>'
    f'<div style="font-size:20px;color:#555;margin-top:2px;">Sample: {sample_name}</div>'
)
if extra_info:
    kv_rows = "".join(
        f'<tr><td style="padding:2px 12px 2px 0;color:#555;">{k}</td>'
        f'<td style="padding:2px 0;"><code>{v}</code></td></tr>'
        for k, v in extra_info.items()
    )
    header_html += (
        f'<table style="margin:10px 0 0 0;border-collapse:collapse;font-size:14px;">' f"{kv_rows}" f"</table>"
    )
header_html += "</div>"
ipy_display(HTML(header_html))

## Table of contents

- [1. Mixed reads stats](#1.-Mixed-reads-stats)
  - [1.1 Strand-ratio category percentages](#1.1-Strand-ratio-category-percentages)
  - [1.2 Strand-ratio category plot](#1.2-Strand-ratio-category-plot)
  - [1.3 Start / end tag concordance](#1.3-Start-/-end-tag-concordance)
  - [1.4 Overall strand-ratio distribution](#1.4-Overall-strand-ratio-distribution)
  - [1.5 Strand-ratio by end-tag category](#1.5-Strand-ratio-by-end-tag-category)
- [2. Trimming stats](#2.-Trimming-stats)
  - [2.1 Failure-code metrics](#2.1-Failure-code-metrics)
  - [2.2 Detailed failure codes](#2.2-Detailed-failure-codes)
- [3. Sequencing QC stats](#3.-Sequencing-QC-stats)
  - [3.1 Sorter stats](#3.1-Sorter-stats)
  - [3.2 Read-length distribution](#3.2-Read-length-distribution)
  - [3.3 Read length by start-tag category](#3.3-Read-length-by-start-tag-category)
- [4. Technical details](#4.-Technical-details)
  - [4.1 Run inputs](#4.1-Run-inputs)
  - [4.2 All stored statistics tables](#4.2-All-stored-statistics-tables)

# 1. Mixed reads stats

## 1.1 Strand-ratio category percentages

In [ ]:
_sample_header()
stats_shortlist = pd.read_hdf(statistics_h5, key="stats_shortlist")
if isinstance(stats_shortlist, pd.Series):
    stats_shortlist = stats_shortlist.to_frame()
ipy_display(stats_shortlist.style.format("{:.2f}"))
_caption(
    "Table 1.1",
    "ppmSeq mixed-read metrics. PCT_MIXED_start_tag is the fraction of reads whose start-loop tag "
    "is MIXED; PCT_MIXED_both_tags is the fraction whose start- and end-loop tags are both MIXED. "
    "The PCT_*_both_tags_where_endreached rows restrict the denominator to reads whose end loop "
    "was reached (tm contains 'A'). PCT_read_end_unreached is the fraction of reads where the "
    "adapter was not reached so no end-tag call is available.",
)

## 1.2 Strand-ratio category plot

In [ ]:
_show_image(
    strand_ratio_category_png,
    "Figure 1.1",
    "Proportion of reads in each strand-ratio category (MIXED / MINUS / PLUS / UNDETERMINED), "
    "plotted separately for the start- and end-loops. The end-loop bars restrict the denominator "
    "to reads that reached the end loop.",
)

## 1.3 Start / end tag concordance

In [ ]:
_show_image(
    strand_ratio_category_concordance_png,
    "Figure 1.2",
    "Concordance heatmap: joint distribution of the start- and end-tag categories across all "
    "reads (top panel) and across only reads where the end was reached (bottom panel). On-diagonal "
    "cells are concordant reads; off-diagonal cells include genuine DISCORDANT reads and reads "
    "where at least one end is UNDETERMINED.",
)

## 1.4 Overall strand-ratio distribution

In [ ]:
_show_image(
    sr_hist_png,
    "Figure 1.3",
    "Strand-ratio distribution across all reads, as a line plot over 0.01-wide bin centers. "
    "Dashed vertical lines at 0.2 and 0.8 mark the --compare cascade thresholds; values outside "
    "[0, 1] are clipped to the edge bins (np.clip).",
)

## 1.5 Strand-ratio by end-tag category

In [ ]:
_show_image(
    sr_by_et_png,
    "Figure 1.4",
    "Strand-ratio distribution faceted by end-tag category (PLUS / MINUS / MIXED / UNDETERMINED) "
    "on reads whose end was reached. Each panel is normalised to 100% within its category so the "
    "shapes are directly comparable.",
)

# 2. Trimming stats

## 2.1 Failure-code metrics

In [ ]:
if trimmer_failure_codes_csv is not None:
    with pd.HDFStore(statistics_h5) as s:
        available_keys = set(s.keys())
    if "/failure_codes_metrics" in available_keys:
        _sample_header()
        df_fc = pd.read_hdf(statistics_h5, key="failure_codes_metrics") / 100.0
        ipy_display(df_fc.style.format("{:.4%}"))
        _caption(
            "Table 2.1",
            "Trimmer failure-rate metrics, computed as failed_read_count / total_read_count. "
            "PCT_failed_adapter_dimers is the fraction of reads Trimmer rejected because the insert "
            "was too short (typically adapter-dimers). PCT_failed_unrecognized_start_stem is the "
            "fraction where the start stem sequence could not be matched. "
            "PCT_failed_unrecognized_start_loop is the fraction where the start loop sequence "
            "was too long to identify. PCT_failed_total is the overall fraction of reads Trimmer "
            "rejected.",
        )
else:
    ipy_display(HTML("<p><i>No trimmer failure codes CSV supplied.</i></p>"))

## 2.2 Detailed failure codes

In [ ]:
if trimmer_failure_codes_csv is not None:
    with pd.HDFStore(statistics_h5) as s:
        available_keys = set(s.keys())
    if "/trimmer_failure_codes" in available_keys:
        _sample_header()
        df_codes = pd.read_hdf(statistics_h5, key="trimmer_failure_codes").copy()
        if "PCT_failure" in df_codes.columns:
            # The CSV's PCT_failure column is already a percentage (not a fraction), so
            # divide by 100 before the `{:.4%}` formatter multiplies it again.
            df_codes["PCT_failure"] = df_codes["PCT_failure"] / 100.0
            ipy_display(df_codes.style.format({"PCT_failure": "{:.4%}"}))
        else:
            ipy_display(df_codes)
        _caption(
            "Table 2.2",
            "Per-segment, per-reason Trimmer failure breakdown. Rows are indexed by "
            "(segment, reason). failed_read_count is the number of reads that failed that check; "
            "total_read_count is the denominator; PCT_failure is the ratio.",
        )

# 3. Sequencing QC stats

## 3.1 Sorter stats

In [ ]:
if sorter_stats_csv is not None:
    with pd.HDFStore(statistics_h5) as s:
        available_keys = set(s.keys())
    if "/sorter_stats" in available_keys:
        sorter_stats = pd.read_hdf(statistics_h5, key="sorter_stats")
        if isinstance(sorter_stats, pd.Series):
            sorter_stats = sorter_stats.to_frame()
        if len(sorter_stats):
            _sample_header()
            ipy_display(sorter_stats.style.format("{:.2f}"))
            _caption(
                "Table 3.1",
                "Sorter application-QC metrics, from the sorter stats CSV. PCT_PF_aligned / "
                "PCT_PF_Reads_aligned are alignment yields; PCT_PF_Q20_bases / PCT_PF_HQ_aligned "
                "summarise base-quality; the PF_Barcode_reads row is the total read count.",
            )
else:
    ipy_display(HTML("<p><i>No sorter stats CSV supplied.</i></p>"))

## 3.2 Read-length distribution

In [ ]:
_show_image(
    read_length_png,
    "Figure 3.1",
    "Read-length distribution across all subsampled reads, from the per-read SAM query lengths. "
    "The x-axis is clipped at the 99.5th percentile so the long tail doesn't compress the bulk "
    "of the distribution.",
)

## 3.3 Read length by start-tag category

In [ ]:
_show_image(
    read_length_by_st_png,
    "Figure 3.2",
    "Read-length distribution faceted by start-tag category (PLUS / MINUS / MIXED). "
    "Useful for checking whether MIXED reads are systematically shorter or longer than PLUS / MINUS "
    "reads, which would suggest an insert-size bias in the strand-ratio call. UNDETERMINED reads "
    "are excluded (they are effectively never present at this point in the pipeline).",
)

# 4. Technical details

## 4.1 Run inputs

In [ ]:
_sample_header()
run_info_rows = [
    ("Sample name", sample_name),
    ("Statistics HDF5", statistics_h5),
    ("Trimmer failure codes CSV", trimmer_failure_codes_csv or "(none supplied)"),
    ("Sorter stats CSV", sorter_stats_csv or "(none supplied)"),
]
for k, v in extra_info.items():
    run_info_rows.append((k, v))
run_info_df = pd.DataFrame(run_info_rows, columns=["field", "value"]).set_index("field")
# Long paths (e.g. "Trimmer failure codes CSV" values) otherwise get visually truncated
# with an ellipsis — force word-wrap so the full path is readable.
html = run_info_df.to_html()
ipy_display(
    HTML(
        "<style>table.dataframe td, table.dataframe th { "
        "word-break: break-all; white-space: normal; max-width: 700px; "
        "text-align: left; vertical-align: top; "
        "}</style>" + html
    )
)
_caption(
    "Table 4.1",
    "Run inputs for this QC report: sample identifier, auxiliary CSV / HDF5 files, and any "
    "free-form --extra-arg KEY=VALUE pairs (e.g. pipeline version).",
)

## 4.2 All stored statistics tables

In [ ]:
with pd.HDFStore(statistics_h5) as s:
    keys = sorted(s.keys())
shown = 0
for key in keys:
    short = key.lstrip("/")
    # Skip what's already shown or internal-only.
    if short in (
        "stats_shortlist",
        "sorter_stats",
        "failure_codes_metrics",
        "trimmer_failure_codes",
        "subsampled_reads",
        "keys_to_convert",
    ):
        continue
    shown += 1
    _sample_header()
    ipy_display(HTML(f'<p style="font-weight:bold;margin:4px 0;">{short}</p>'))
    ipy_display(pd.read_hdf(statistics_h5, key=key))
if shown == 0:
    ipy_display(HTML("<p><i>No additional stored tables.</i></p>"))
_caption(
    "Table 4.2",
    "All additional statistics tables stored in the HDF5 bundle beyond those shown in Sections 1-3. "
    "Typical entries: strand_ratio_category_counts (raw read counts per category), "
    "strand_ratio_category_norm (same as fractions), strand_ratio_category_concordance (pairwise "
    "start/end concordance), strand_ratio_category_consensus (per-read consensus category shares).",
)

In [ ]:
_show_logo(logo_file, align="left")